# Ejercicios Spark DataFrames

Vamos a practicar un poco con tus nuevas habilidades de Spark DataFrame, se te harán algunas preguntas básicas sobre algunos datos del mercado de valores, en este caso Walmart Stock de los años 2012-2017.

Responde a las preguntas y completa las tareas de abajo.

#### ¡Utiliza el archivo walmart_stock.csv para responder y completar las tareas siguientes!

#### Iniciar una sesión de Spark

#### Preparar entorno y arrancar Spark

Esta celda esta pensada para ejecutarse de principio a fin en local o en Google Colab. Si PySpark ya esta instalado, no lo reinstala; si falta y estas en Colab, instala Java 17 y PySpark antes de crear la sesion.


In [21]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
JAVA_HOME = "/usr/lib/jvm/java-17-openjdk-amd64"
PYSPARK_VERSION = "4.0.1"


def run_command(command):
    print("Ejecutando:", " ".join(command))
    subprocess.run(command, check=True)


def java_available():
    if Path(JAVA_HOME).exists():
        return True
    return subprocess.run(["which", "java"], capture_output=True).returncode == 0


if IN_COLAB:
    if not java_available():
        print("Java 17 no esta disponible. Instalando Java 17 para Colab...")
        run_command(["apt-get", "update", "-qq"])
        run_command(["apt-get", "install", "-y", "openjdk-17-jdk-headless", "-qq"])
    else:
        print("Java disponible. Se reutiliza el entorno actual.")

    os.environ["JAVA_HOME"] = JAVA_HOME
    os.environ["PATH"] += f":{JAVA_HOME}/bin"

    if importlib.util.find_spec("pyspark") is None:
        print("PySpark no esta instalado. Instalando PySpark para Colab...")
        run_command([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", f"pyspark=={PYSPARK_VERSION}"])
    else:
        print("PySpark ya esta instalado. Se reutiliza el entorno actual.")
else:
    if importlib.util.find_spec("pyspark") is None:
        raise ModuleNotFoundError(
            "PySpark no esta instalado. En local instala las dependencias con: "
            "python -m pip install -r requirements.txt"
        )
    print("Entorno local con PySpark disponible.")

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Practica Spark DataFrame - Walmart Stock")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
spark.range(10).show()
print("Spark listo")

Java disponible. Se reutiliza el entorno actual.
PySpark ya esta instalado. Se reutiliza el entorno actual.
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+

Spark listo


#### Cargar el archivo CSV de Walmart Stock, hacer que Spark infiera los tipos de datos.

In [22]:
df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .csv("walmart_stock.csv"))

df.show(5)

+----------+------------------+---------+---------+------------------+--------+------------------+
|      Date|              Open|     High|      Low|             Close|  Volume|         Adj Close|
+----------+------------------+---------+---------+------------------+--------+------------------+
|2012-01-03|         59.970001|61.060001|59.869999|         60.330002|12668800|52.619234999999996|
|2012-01-04|60.209998999999996|60.349998|59.470001|59.709998999999996| 9593300|         52.078475|
|2012-01-05|         59.349998|59.619999|58.369999|         59.419998|12768200|         51.825539|
|2012-01-06|         59.419998|59.450001|58.869999|              59.0| 8069400|          51.45922|
|2012-01-09|         59.029999|59.549999|58.919998|             59.18| 6679300|51.616215000000004|
+----------+------------------+---------+---------+------------------+--------+------------------+
only showing top 5 rows


#### ¿Cuáles son los nombres de las columnas?

In [23]:
df.columns

['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close']

#### ¿Qué aspecto tiene el esquema?

In [24]:
df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: integer (nullable = true)
 |-- Adj Close: double (nullable = true)



#### Imprime las 5 primeras columnas.

In [25]:
for row in df.head(5):
    print(row)
    print()

Row(Date=datetime.date(2012, 1, 3), Open=59.970001, High=61.060001, Low=59.869999, Close=60.330002, Volume=12668800, Adj Close=52.619234999999996)

Row(Date=datetime.date(2012, 1, 4), Open=60.209998999999996, High=60.349998, Low=59.470001, Close=59.709998999999996, Volume=9593300, Adj Close=52.078475)

Row(Date=datetime.date(2012, 1, 5), Open=59.349998, High=59.619999, Low=58.369999, Close=59.419998, Volume=12768200, Adj Close=51.825539)

Row(Date=datetime.date(2012, 1, 6), Open=59.419998, High=59.450001, Low=58.869999, Close=59.0, Volume=8069400, Adj Close=51.45922)

Row(Date=datetime.date(2012, 1, 9), Open=59.029999, High=59.549999, Low=58.919998, Close=59.18, Volume=6679300, Adj Close=51.616215000000004)



#### Utiliza describe() para conocer el DataFrame.

In [26]:
df.describe().show()

+-------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|summary|              Open|             High|              Low|            Close|           Volume|        Adj Close|
+-------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|  count|              1258|             1258|             1258|             1258|             1258|             1258|
|   mean| 72.35785375357709|72.83938807631165| 71.9186009594594|72.38844998012726|8222093.481717011|67.23883848728146|
| stddev|  6.76809024470826|6.768186808159218|6.744075756255496|6.756859163732991|  4519780.8431556|6.722609449996857|
|    min|56.389998999999996|        57.060001|        56.299999|        56.419998|          2094900|        50.363689|
|    max|         90.800003|        90.970001|            89.25|        90.470001|         80898100|84.91421600000001|
+-------+------------------+-----------------+--

#### Hay demasiados decimales para la media y el stddev en describe(). Formatea los números para que sólo se muestren hasta dos decimales. Presta atención a los tipos de datos que devuelve .describe()

In [ ]:
summary = df.describe()
summary.printSchema()

In [ ]:
from pyspark.sql.functions import format_number

formatted_summary = summary.select(
    summary["summary"],
    format_number(summary["Open"].cast("double"), 2).alias("Open"),
    format_number(summary["High"].cast("double"), 2).alias("High"),
    format_number(summary["Low"].cast("double"), 2).alias("Low"),
    format_number(summary["Close"].cast("double"), 2).alias("Close"),
    format_number(summary["Volume"].cast("double"), 0).alias("Volume")
)

In [ ]:
formatted_summary.show()

#### Crea un nuevo dataframe con una columna llamada HV Ratio que es la relación entre el precio máximo y el volumen de las acciones negociadas durante un día.

In [29]:
df_hv = df.withColumn("HV Ratio", df["High"] / df["Volume"])
df_hv.select("HV Ratio").show()

+--------------------+
|            HV Ratio|
+--------------------+
|4.819714653321546E-6|
|6.290848613094555E-6|
|4.669412994783916E-6|
|7.367338463826307E-6|
|8.915604778943901E-6|
|8.644477436914568E-6|
|9.351828421515645E-6|
| 8.29141562102703E-6|
|7.712212102001476E-6|
|7.071764823529412E-6|
|1.015495466386981E-5|
|6.576354146362592...|
| 5.90145296180676E-6|
|8.547679455011844E-6|
|8.420709512685392E-6|
|1.041448341728929...|
|8.316075414862431E-6|
|9.721183814992126E-6|
|8.029436027707578E-6|
|6.307432259386365E-6|
+--------------------+
only showing top 20 rows


#### ¿Qué día hubo el pico máximo en el precio?

In [30]:
df.orderBy(df["High"].desc()).select("Date", "High").show(1)

# Si se quiere solo el valor de la fecha:
df.orderBy(df["High"].desc()).head()["Date"]

+----------+---------+
|      Date|     High|
+----------+---------+
|2015-01-13|90.970001|
+----------+---------+
only showing top 1 row


datetime.date(2015, 1, 13)

#### ¿Cuál es la media de la columna Close?

In [31]:
from pyspark.sql.functions import avg

df.select(avg("Close").alias("Media Close")).show()

+-----------------+
|      Media Close|
+-----------------+
|72.38844998012726|
+-----------------+



#### ¿Cuál es el máximo y el mínimo de la columna Volumen?

In [ ]:
from pyspark.sql.functions import max, min

In [ ]:
df.select(
    max("Volume").alias("Max Volume"),
    min("Volume").alias("Min Volume")
).show()

#### ¿Cuántos días estuvo el cierre por debajo de los 60 dólares?

In [33]:
df.filter(df["Close"] < 60).count()

81

#### ¿Qué porcentaje de veces el Máximo fue superior a 80 dólares?
#### En otras palabras, (Número de días de máximos>80)/(Días totales en el conjunto de datos)

In [34]:
dias_high_mayor_80 = df.filter(df["High"] > 80).count()
total_dias = df.count()
(dias_high_mayor_80 / total_dias) * 100

9.141494435612083

#### ¿Cuál es la correlación de Pearson entre High y Volume?

In [ ]:
from pyspark.sql.functions import corr

df.select(corr("High", "Volume").alias("Correlacion High-Volume")).show()

#### ¿Cuál es el valor máximo de High por año?

In [ ]:
from pyspark.sql.functions import year

df_year = df.withColumn("Year", year(df["Date"]))
df_year.groupBy("Year").max("High").orderBy("Year").show()

#### ¿Cuál es el cierre medio de cada mes del calendario?
#### En otras palabras, a lo largo de todos los años, ¿cuál es el precio medio de cierre para enero, febrero, marzo, etc.? Su resultado tendrá un valor para cada uno de estos meses.

In [ ]:
from pyspark.sql.functions import month

df_month = df.withColumn("Month", month(df["Date"]))
df_month.groupBy("Month").avg("Close").orderBy("Month").show()